<a href="https://colab.research.google.com/github/brianellis1997/Sarcasm_Detection/blob/main/CNN_Test_Accuracy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !git clone https://github.com/brianellis1997/Sarcasm_Detection.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
test_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Test_Balanced.csv')

In [ ]:
test_bal

In [ ]:
train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/train_bal_2.csv')

In [ ]:
train_bal

# CNN Best Model on Test Data

In [ ]:
import gensim
from gensim.models.phrases import Phraser, Phrases
from gensim.models.word2vec import Word2Vec
word2vec = gensim.models.Word2Vec.load('/content/drive/MyDrive/Sarcasm_Data/word2vec.model')

In [ ]:
max_words = 10000  # Maximum number of words to keep based on word frequency
max_seq_length = 100  # Maximum length of sequences

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, concatenate, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model

tokenizer = Tokenizer(num_words=max_words, lower=False)
tokenizer.fit_on_texts(train_bal['comment'] + ' ' + train_bal['parent_comment'])

In [ ]:
import numpy as np
from keras.layers import Embedding
# Initialize the embedding matrix
embedding_dim = word2vec.vector_size
embedding_matrix = np.zeros((max_words, embedding_dim))

# Populate the embedding matrix using the tokenizer's word_index
for word, i in tokenizer.word_index.items():
    if i < max_words:  # Ensure you don't exceed the max_words limit
        if word in word2vec.wv:
            embedding_matrix[i] = word2vec.wv[word]

In [ ]:
y_test = test_bal['label']

In [ ]:
y_train = train_bal['label']

In [ ]:
X_text_comment_seq_test = tokenizer.texts_to_sequences(test_bal['comment'])
X_text_parent_comment_seq_test = tokenizer.texts_to_sequences(test_bal['parent_comment'])

In [ ]:
X_text_comment_seq= tokenizer.texts_to_sequences(train_bal['comment'])
X_text_parent_comment_seq = tokenizer.texts_to_sequences(train_bal['parent_comment'])

In [ ]:
X_text_comment_pad_test = pad_sequences(X_text_comment_seq_test, maxlen=max_seq_length)
X_text_parent_comment_pad_test = pad_sequences(X_text_parent_comment_seq_test, maxlen=max_seq_length)

In [ ]:
X_text_comment_pad = pad_sequences(X_text_comment_seq, maxlen=max_seq_length)
X_text_parent_comment_pad = pad_sequences(X_text_parent_comment_seq, maxlen=max_seq_length)

In [ ]:
pip install keras-tuner

In [ ]:
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout, Bidirectional, LSTM
from keras.models import Model
from keras import optimizers
from keras_tuner import HyperModel

def build_model(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv1_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=64, step=32),
                   kernel_size=hp.Choice('conv2_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)

    bi_lstm = Bidirectional(LSTM(hp.Int('lstm_units', min_value=64, max_value=256, step=64)))(conv2)

    # Because we've added a Bi-LSTM layer, we no longer need to flatten the conv2 output
    bi_lstm = Dropout(0.5)(bi_lstm)

    dense = Dense(hp.Int('dense_units', min_value=64, max_value=128, step=32), activation='relu')(bi_lstm)
    output = Dense(1, activation='sigmoid')(dense)

    model = Model(inputs=[comment_input, parent_comment_input], outputs=output)

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model


In [ ]:
from kerastuner.tuners import RandomSearch

tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='my_dir',
    project_name='hparam_tuning4'
)

tuner.search_space_summary()

tuner.search(x=[X_text_comment_pad, X_text_parent_comment_pad],
             y=y_train,
             epochs=4,
             batch_size=128,
             validation_split=0.10)



CNN_Text_BiLSTM = tuner.get_best_models(num_models=1)[0]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text_BiLSTM .predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)